In [1]:
import duckdb
import pandas as pd

DB_PATH = "../data/processed/safety.duckdb"

con = duckdb.connect(DB_PATH, read_only=True)

df = con.execute("""
    select *
    from ml_injury_features
""").df()

df.head()

,INCIDENT_ID,EVENT_DATE,event_year,event_month,event_day_of_week,State,PRIMARY_NAICS,naics_2_digit,naics_3_digit,NATURE_TITLE,BODY_PART_TITLE,EVENT_TITLE,SOURCE_TITLE,SECONDARY_SOURCE_TITLE,high_severity_outcome
0,2015010015,2015-01-01,2015,1,4,NEW YORK,922140,92,922,Fractures,Lower leg(s),Injured by physical contact with person while ...,Co-worker,Inmate or detainee in custody,0
1,2015010016,2015-01-01,2015,1,4,WISCONSIN,339999,33,339,Second degree heat (thermal) burns,"Leg(s), n.e.c.","Ignition of vapors, gases, or liquids","Welding, cutting, and blow torches",None,0
2,2015010018,2015-01-01,2015,1,4,PENNSYLVANIA,484121,48,484,"Traumatic injuries and disorders, unspecified",Nonclassifiable,Other fall to lower level less than 6 feet,"Semi, tractor-trailer, tanker truck",Ladders-fixed,0
3,2015010019,2015-01-01,2015,1,4,GEORGIA,424490,42,424,"Soreness, pain, hurt-nonspecified injury","Leg(s), unspecified",Caught in or compressed by equipment or object...,Pallet jack-powered,"Truck-motorized freight hauling and utility, u...",0
4,2015010020,2015-01-01,2015,1,4,WISCONSIN,326122,32,326,Fractures,"Finger(s), fingernail(s), n.e.c.",Caught in running equipment or machinery durin...,"Metal, woodworking, and special material machi...",None,0


In [2]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

df.info()

Rows: 105,318
Columns: 15
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105318 entries, 0 to 105317
Data columns (total 15 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   INCIDENT_ID             105318 non-null  object        
 1   EVENT_DATE              105318 non-null  datetime64[us]
 2   event_year              105318 non-null  int64         
 3   event_month             105318 non-null  int64         
 4   event_day_of_week       105318 non-null  int64         
 5   State                   105318 non-null  object        
 6   PRIMARY_NAICS           105316 non-null  object        
 7   naics_2_digit           105316 non-null  object        
 8   naics_3_digit           105316 non-null  object        
 9   NATURE_TITLE            105318 non-null  object        
 10  BODY_PART_TITLE         105318 non-null  object        
 11  EVENT_TITLE             105318 non-null  object        
 12  SOUR

In [3]:
df["high_severity_outcome"].value_counts(dropna=False)

high_severity_outcome
0    77512
1    27806
Name: count, dtype: int64

In [4]:
df["high_severity_outcome"].value_counts(normalize=True)

high_severity_outcome
0    0.735981
1    0.264019
Name: proportion, dtype: float64

In [5]:
df["NATURE_TITLE"].value_counts().head(30)

NATURE_TITLE
Fractures                                                              28480
Amputations                                                            23397
Soreness, pain, hurt-nonspecified injury                                6510
 Fractures                                                              6085
Cuts, lacerations                                                       4263
 Amputations, avulsions, enucleations  unspecified                      2791
Intracranial injuries, unspecified                                      1799
Traumatic injuries and disorders, unspecified                           1729
Heat (thermal) burns, unspecified                                       1722
Crushing injuries                                                       1722
Internal injuries to organs and blood vessels of the trunk              1514
 Amputations involving bone loss                                        1464
Puncture wounds, except gunshot wounds                         

In [14]:
df["NATURE_TITLE"]

0                                                 Fractures
1                        Second degree heat (thermal) burns
2             Traumatic injuries and disorders, unspecified
3                  Soreness, pain, hurt-nonspecified injury
4                                                 Fractures
                                ...                        
105313     Closed trauma involving internal organs, majo...
105314         Traumatic injuries or exposures  unspecified
105315                      Amputations involving bone loss
105316                        Sprains, strains, minor tears
105317     Cuts, lacerations, punctures without injury t...
Name: NATURE_TITLE, Length: 105318, dtype: object

In [13]:
df["NATURE_TITLE"].str.contains(
        r"amput|eye",
        case=False,
        na=False
    )

0         False
1         False
2         False
3         False
4         False
          ...  
105313    False
105314    False
105315     True
105316    False
105317    False
Name: NATURE_TITLE, Length: 105318, dtype: bool

In [11]:
df.loc[
    df["NATURE_TITLE"].str.contains(
        r"amput|eye",
        case=False,
        na=False
    ),
    ["NATURE_TITLE", "high_severity_outcome"]
].value_counts()

NATURE_TITLE                                               high_severity_outcome
Amputations                                                1                        23060
 Amputations, avulsions, enucleations  unspecified         1                         2758
 Amputations involving bone loss                           1                         1452
Amputations                                                0                          337
 Amputations                                               1                           50
Amputations, avulsions, enucleations, n.e.c.               1                           34
 Amputations, avulsions, enucleations  unspecified         0                           33
 Eye abrasion(s), irritation  except chemical or allergic  0                           12
 Amputations involving bone loss                           0                           12
Amputations, avulsions, enucleations, unspecified          1                            3
                   